# 03 — Sentiment model experiments

This notebook is intentionally structured as one controlled experiment. Every model uses the same game-held-out split and the same evaluation metrics, so comparisons are meaningful. Run the cells top-to-bottom. The Transformer experiment remains separate because it needs additional packages and usually a GPU.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'reviews_clean.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
RANDOM_STATE = 42
RUN_XGBOOST = False  # Change to True only after the fast baselines finish.
MAX_FEATURES = 50_000  # Safer default for notebook memory.
SCOPE_EXCLUSIONS_PATH = PROJECT_ROOT / 'data' / 'game_scope_exclusions.csv'
KEEP_FITTED_MODELS = False  # Keep predictions/metrics, not every large vectorizer.

## 1. Load data and define the modeling population

In [2]:
reviews = pd.read_csv(DATA_PATH)
if SCOPE_EXCLUSIONS_PATH.exists():
    excluded_ids = pd.read_csv(SCOPE_EXCLUSIONS_PATH, dtype={'AppID':'string'}).loc[lambda d: d.scope_status.eq('excluded'), 'AppID'].astype(str).tolist()
    reviews = reviews[~reviews.AppID.astype(str).isin(excluded_ids)].copy()
    print('Excluded from indie scope:', excluded_ids)
reviews = reviews[reviews['review_clean'].fillna('').str.strip().ne('')].copy()
reviews = reviews.drop_duplicates('recommendationid')
reviews['target'] = reviews['voted_up'].astype(int)
print(f'{len(reviews):,} usable reviews across {reviews.AppID.nunique():,} games')
display(reviews[['target', 'AppID']].describe(include='all'))
display(reviews['target'].value_counts(normalize=True).rename('share'))

Excluded from indie scope: ['252950']
210,250 usable reviews across 237 games


,target,AppID
count,210250.000000,2.102500e+05
mean,0.899144,1.602912e+06
std,0.301139,1.212661e+06
min,0.000000,3.590000e+03
25%,1.000000,5.415700e+05
50%,1.000000,1.243830e+06
75%,1.000000,2.670630e+06
max,1.000000,4.800590e+06


target
1    0.899144
0    0.100856
Name: share, dtype: float64

## 2. One game-held-out split

Reviews from the same game can share vocabulary, names, and recurring phrases. Holding out complete games gives a more honest estimate of how well the model transfers to a new game.

In [3]:
X = reviews['review_clean']
y = reviews['target']
if reviews.AppID.nunique() >= 5:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=reviews['AppID']))
    split_name = 'game-held-out'
else:
    train_idx, test_idx = train_test_split(np.arange(len(reviews)), test_size=0.20, stratify=y, random_state=RANDOM_STATE)
    split_name = 'stratified review split'
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(split_name, '| train:', len(train_idx), '| test:', len(test_idx))
print('train games:', reviews.iloc[train_idx].AppID.nunique(), '| test games:', reviews.iloc[test_idx].AppID.nunique())
print('overlapping games:', len(set(reviews.iloc[train_idx].AppID) & set(reviews.iloc[test_idx].AppID)))

game-held-out | train: 165917 | test: 44333
train games: 189 | test games: 48
overlapping games: 0


## 3. Shared experiment runner

Balanced accuracy and macro F1 are primary metrics because Steam recommendations are highly imbalanced. ROC-AUC is reported when the estimator provides a continuous score.

In [4]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    if hasattr(model, 'decision_function'):
        score = model.decision_function(X_test)
    elif hasattr(model, 'predict_proba'):
        score = model.predict_proba(X_test)[:, 1]
    else:
        score = pred
    row = {'model': name, 'accuracy': accuracy_score(y_test, pred),
           'balanced_accuracy': balanced_accuracy_score(y_test, pred),
           'macro_f1': f1_score(y_test, pred, average='macro', zero_division=0),
           'positive_f1': f1_score(y_test, pred, zero_division=0),
           'precision': precision_score(y_test, pred, zero_division=0),
           'recall': recall_score(y_test, pred, zero_division=0),
           'roc_auc': roc_auc_score(y_test, score)}
    print(name)
    print(classification_report(y_test, pred, target_names=['Negative', 'Positive'], zero_division=0))
    return row, model, pred

## 4. Fast baselines

Run these first. Logistic regression and Linear SVM are strong, standard sparse-text baselines and are usually more natural choices than tree boosting over a very wide TF-IDF matrix.

In [5]:
experiments = {}
results = []
majority = DummyClassifier(strategy='most_frequent')
row, model, pred = evaluate_model('majority', majority)
results.append(row); experiments['majority'] = pred
del model

majority
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00      4521
    Positive       0.90      1.00      0.95     39812

    accuracy                           0.90     44333
   macro avg       0.45      0.50      0.47     44333
weighted avg       0.81      0.90      0.85     44333



In [6]:
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=MAX_FEATURES, sublinear_tf=True)
logistic = Pipeline([('tfidf', word_tfidf), ('classifier', LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=1))])
row, model, pred = evaluate_model('logistic + word TF-IDF', logistic)
results.append(row); experiments['logistic_word'] = pred
del model, logistic, word_tfidf

logistic + word TF-IDF
              precision    recall  f1-score   support

    Negative       0.53      0.85      0.65      4521
    Positive       0.98      0.91      0.95     39812

    accuracy                           0.91     44333
   macro avg       0.76      0.88      0.80     44333
weighted avg       0.94      0.91      0.92     44333



In [7]:
svm = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=MAX_FEATURES, sublinear_tf=True)), ('classifier', LinearSVC(class_weight='balanced'))])
row, model, pred = evaluate_model('Linear SVM + word TF-IDF', svm)
results.append(row); experiments['svm_word'] = pred
del model, svm

Linear SVM + word TF-IDF
              precision    recall  f1-score   support

    Negative       0.58      0.78      0.66      4521
    Positive       0.97      0.93      0.95     39812

    accuracy                           0.92     44333
   macro avg       0.78      0.86      0.81     44333
weighted avg       0.93      0.92      0.92     44333



In [8]:
char_logistic = Pipeline([('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3, max_features=MAX_FEATURES, sublinear_tf=True)), ('classifier', LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=1))])
row, model, pred = evaluate_model('logistic + character TF-IDF', char_logistic)
results.append(row); experiments['logistic_char'] = pred
del model, char_logistic

logistic + character TF-IDF
              precision    recall  f1-score   support

    Negative       0.49      0.86      0.62      4521
    Positive       0.98      0.90      0.94     39812

    accuracy                           0.89     44333
   macro avg       0.74      0.88      0.78     44333
weighted avg       0.93      0.89      0.91     44333



## 5. XGBoost comparison (optional)

This preserves the original requested baseline. It is intentionally opt-in because XGBoost over a 100k-dimensional sparse matrix can be much slower and does not use word order or context.

In [9]:
if RUN_XGBOOST:
    from xgboost import XGBClassifier
    xgb = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=MAX_FEATURES, sublinear_tf=True)), ('classifier', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, objective='binary:logistic', eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=1, scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1)))])
    row, model, pred = evaluate_model('XGBoost + word TF-IDF', xgb)
    results.append(row); experiments['xgboost'] = pred
    del model, xgb
else:
    print('Skipped: set RUN_XGBOOST = True to run this optional comparison.')

Skipped: set RUN_XGBOOST = True to run this optional comparison.


## 6. Compare and save results

In [10]:
results_df = pd.DataFrame(results).sort_values('balanced_accuracy', ascending=False)
display(results_df)
results_df.to_csv(OUTPUT_DIR / 'model_results.csv', index=False)
with open(OUTPUT_DIR / 'model_experiment_config.json', 'w', encoding='utf-8') as f:
    json.dump({'split': split_name, 'random_state': RANDOM_STATE, 'max_features': MAX_FEATURES, 'train_rows': len(train_idx), 'test_rows': len(test_idx)}, f, indent=2)

,model,accuracy,balanced_accuracy,macro_f1,positive_f1,precision,recall,roc_auc
1,logistic + word TF-IDF,0.908014,0.884081,0.800677,0.946947,0.982188,0.914146,0.953011
3,logistic + character TF-IDF,0.893849,0.878252,0.780426,0.938238,0.982437,0.897845,0.949249
2,Linear SVM + word TF-IDF,0.919022,0.857171,0.808261,0.953991,0.973911,0.934869,0.944030
0,majority,0.898022,0.500000,0.473136,0.946271,0.898022,1.000000,0.500000


## 7. Per-game performance and error analysis

Aggregate scores can hide games where the model fails completely. Use the best model from the table below, then inspect performance by held-out game and representative errors.

In [11]:
best_name = results_df.iloc[0]['model']
best_key = {'majority': 'majority', 'logistic + word TF-IDF': 'logistic_word', 'Linear SVM + word TF-IDF': 'svm_word', 'logistic + character TF-IDF': 'logistic_char', 'XGBoost + word TF-IDF': 'xgboost'}[best_name]
best_pred = experiments[best_key]
held_out = reviews.iloc[test_idx][['AppID', 'Name', 'review_clean', 'target']].copy()
held_out['prediction'] = best_pred
per_game = held_out.groupby(['AppID', 'Name']).apply(lambda g: pd.Series({'reviews': len(g), 'balanced_accuracy': balanced_accuracy_score(g.target, g.prediction) if g.target.nunique() > 1 else np.nan, 'accuracy': accuracy_score(g.target, g.prediction)}), include_groups=False).reset_index()
display(per_game.sort_values('balanced_accuracy').head(20))
errors = held_out[held_out.target != held_out.prediction].copy()
display(errors[['Name', 'target', 'prediction', 'review_clean']].head(30))

,AppID,Name,reviews,balanced_accuracy,accuracy
33,2709570,Supermarket Together,32.0,0.700000,0.875000
9,319510,Five Nights at Freddy's,984.0,0.729854,0.888211
21,1256670,Library Of Ruina,985.0,0.756557,0.891371
5,252490,Rust,957.0,0.769116,0.817137
1,223470,POSTAL 2,970.0,0.783682,0.844330
2,227300,Euro Truck Simulator 2,955.0,0.813118,0.939267
8,291550,Brawlhalla,990.0,0.819317,0.869697
16,774171,Muse Dash,972.0,0.830973,0.905350
32,2592160,Dispatch,983.0,0.834030,0.923703
15,747660,Five Nights at Freddy's: Security Breach,984.0,0.842408,0.866870


,Name,target,prediction,review_clean
7016,Bendy and the Dark Revival,1,0,"this is a near-perfect game, but it has a coup..."
7020,Bendy and the Dark Revival,1,0,now although i hate the fact that combat is ho...
7031,Bendy and the Dark Revival,1,0,"i'm only about half way through, but as someon..."
7038,Bendy and the Dark Revival,0,1,"it's very slow. every action has a cutscene, w..."
7056,Bendy and the Dark Revival,1,0,wandering is a terrible sin
7068,Bendy and the Dark Revival,1,0,"masterpiece, but the spoilers: controls on con..."
7070,Bendy and the Dark Revival,1,0,amazing game the only problems i have is that ...
7100,Bendy and the Dark Revival,0,1,ive been wanting to have this game since the d...
7101,Bendy and the Dark Revival,1,0,i lowkey dont know if i hate this game or not....
7108,Bendy and the Dark Revival,0,1,i know this game is suppose to be hard and i d...


In [12]:
# --- REPEATED GAME-HELD-OUT EVALUATION ---
print("Running repeated evaluations across 5 random seeds to verify stability...")

seeds_to_test = [42, 123, 999, 2024, 7]
seed_results = []

for seed in seeds_to_test:
    # 1. Split the data using the current loop's seed
    if reviews.AppID.nunique() >= 5:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
        train_idx_seed, test_idx_seed = next(splitter.split(X, y, groups=reviews['AppID']))
    else:
        train_idx_seed, test_idx_seed = train_test_split(np.arange(len(reviews)), test_size=0.20, stratify=y, random_state=seed)
        
    X_train_seed, X_test_seed = X.iloc[train_idx_seed], X.iloc[test_idx_seed]
    y_train_seed, y_test_seed = y.iloc[train_idx_seed], y.iloc[test_idx_seed]
    
    # 2. Re-initialize and train the pipeline so data doesn't leak between runs
    word_tfidf_seed = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=MAX_FEATURES, sublinear_tf=True)
    logistic_seed = Pipeline([
        ('tfidf', word_tfidf_seed), 
        ('classifier', LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=1))
    ])
    
    logistic_seed.fit(X_train_seed, y_train_seed)
    pred_seed = logistic_seed.predict(X_test_seed)
    
    # 3. Calculate and store the balanced accuracy
    acc = balanced_accuracy_score(y_test_seed, pred_seed)
    seed_results.append({'seed': seed, 'balanced_accuracy': acc})

# Display the final stability report
seed_results_df = pd.DataFrame(seed_results)
display(seed_results_df)
print(f"Average Balanced Accuracy across 5 runs: {seed_results_df['balanced_accuracy'].mean():.4f}")
print(f"Standard Deviation (Variance): ±{seed_results_df['balanced_accuracy'].std():.4f}")

Running repeated evaluations across 5 random seeds to verify stability...


,seed,balanced_accuracy
0,42,0.884081
1,123,0.880753
2,999,0.895337
3,2024,0.881646
4,7,0.890951


Average Balanced Accuracy across 5 runs: 0.8866
Standard Deviation (Variance): ±0.0063
